# FinTorch Time Series Datasets Tutorial

This tutorial demonstrates how to work with FinTorch's standardized time series dataset format and create custom datasets for your own data.

## Overview

FinTorch uses a standardized dictionary format for time series data that:
- Separates different types of features (target, covariates, static)
- Supports multiple time series natively
- Works seamlessly with all FinTorch models
- Provides clear data organization and type safety

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List
from torch.utils.data import DataLoader

# FinTorch imports
from fintorch.datasets.synthetic.simpleSynthetic import SimpleSyntheticDataset
from fintorch.datasets.base import TimeSeriesDataset

print("Libraries imported successfully!")

## 1. Understanding the Data Structure

Let's start by examining the standardized format using the synthetic dataset:

In [ ]:
# Create a simple synthetic dataset
dataset = SimpleSyntheticDataset(
    length=1000,                    # Total time series length
    past_length=24,                 # Historical window
    future_length=6,                # Prediction horizon
    num_series=2,                   # Multiple time series
    num_target_features=1,          # Single target variable
    num_known_cov_features=3,       # 3 known future covariates
    num_unknown_cov_features=2,     # 2 unknown future covariates
    num_static_real_features=2,     # 2 real static features
    num_static_categorical_features=1, # 1 categorical static feature
    static_categorical_cardinalities=[5], # 5 categories
    trend_slope=0.1,
    seasonality_amplitude=2.0,
    noise_level=0.2
)

print(f"Dataset length: {len(dataset)}")
print(f"Time steps (past): {dataset.time_steps}")
print(f"Future steps: {dataset.future_steps}")
print(f"Number of series: {dataset.series_dim}")

In [ ]:
# Examine a single sample
sample = dataset[0]

print("Sample contains the following keys:")
for key, tensor in sample.items():
    print(f"  {key}: shape={tensor.shape}, dtype={tensor.dtype}")

## 2. Visualizing the Data Structure

In [ ]:
# Visualize the time series data
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot past and future targets for both series
for series_idx in range(2):
    ax = axes[series_idx, 0]
    
    # Past target
    past_target = sample["past_target"][:, series_idx, 0]
    future_target = sample["output_target"][:, series_idx, 0]
    
    time_past = range(len(past_target))
    time_future = range(len(past_target), len(past_target) + len(future_target))
    
    ax.plot(time_past, past_target, 'b-', label='Past Target', linewidth=2)
    ax.plot(time_future, future_target, 'r-', label='Future Target', linewidth=2)
    ax.axvline(x=len(past_target)-0.5, color='gray', linestyle='--', alpha=0.7)
    ax.set_title(f'Series {series_idx + 1}: Target Values')
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Value')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Plot known covariates
ax = axes[0, 1]
known_cov = sample["past_covariates_known_future"][:, 0, :3]  # First 3 features, series 0
for i in range(3):
    ax.plot(known_cov[:, i], label=f'Known Cov {i+1}', alpha=0.8)
ax.set_title('Known Future Covariates (Series 1)')
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot unknown covariates
ax = axes[1, 1]
unknown_cov = sample["past_covariates_unknown_future"][:, 0, :2]  # First 2 features, series 0
for i in range(2):
    ax.plot(unknown_cov[:, i], label=f'Unknown Cov {i+1}', alpha=0.8)
ax.set_title('Unknown Future Covariates (Series 1)')
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Working with DataLoaders

In [ ]:
# Create a DataLoader
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Examine a batch
batch = next(iter(dataloader))

print("Batch shapes (with batch dimension added):")
for key, tensor in batch.items():
    print(f"  {key}: {tensor.shape}")

# The batch dimension is now the first dimension
print(f"\nBatch size: {batch['past_target'].shape[0]}")
print(f"Past time steps: {batch['past_target'].shape[1]}")
print(f"Number of series: {batch['past_target'].shape[2]}")
print(f"Target features: {batch['past_target'].shape[3]}")

## 4. Creating a Custom Dataset

Now let's create a custom dataset from scratch. We'll simulate a simple stock price dataset:

In [ ]:
class CustomStockDataset(TimeSeriesDataset):
    def __init__(self, num_stocks=3, total_days=500, past_length=30, future_length=5):
        super().__init__()
        self.past_length = past_length
        self.future_length = future_length
        self.num_stocks = num_stocks
        
        # Generate synthetic stock data
        self.data = self._generate_stock_data(num_stocks, total_days)
        
    def _generate_stock_data(self, num_stocks, total_days):
        np.random.seed(42)
        
        # Generate stock prices with different characteristics
        stock_prices = np.zeros((total_days, num_stocks, 1))
        
        for stock_idx in range(num_stocks):
            # Random walk with trend
            returns = np.random.normal(0.001, 0.02, total_days)
            prices = 100 * np.exp(np.cumsum(returns))
            stock_prices[:, stock_idx, 0] = prices
        
        # Generate time features (known covariates)
        dates = pd.date_range('2020-01-01', periods=total_days, freq='D')
        day_of_week = dates.dayofweek.values
        month = dates.month.values
        is_month_end = dates.is_month_end.astype(int)
        
        # Normalize time features
        known_cov = np.stack([
            day_of_week / 6.0,  # Normalize to [0, 1]
            month / 12.0,       # Normalize to [0, 1]
            is_month_end        # Already 0 or 1
        ], axis=1)
        
        # Expand to all stocks
        known_cov = np.repeat(known_cov[:, np.newaxis, :], num_stocks, axis=1)
        
        # Generate technical indicators (unknown covariates)
        unknown_cov = np.zeros((total_days, num_stocks, 2))
        
        for stock_idx in range(num_stocks):
            prices = stock_prices[:, stock_idx, 0]
            
            # Simple moving average (normalized)
            sma_10 = pd.Series(prices).rolling(10).mean().bfill()
            sma_ratio = prices / sma_10
            
            # Volatility (rolling std)
            volatility = pd.Series(prices).rolling(10).std().fillna(0)
            volatility = volatility / volatility.std()  # Normalize
            
            unknown_cov[:, stock_idx, 0] = sma_ratio
            unknown_cov[:, stock_idx, 1] = volatility
        
        # Generate static features
        # Real features: market cap (log), beta
        static_real = np.array([
            [np.log(1e9), 1.2],   # Stock 0: Large cap, high beta
            [np.log(5e8), 0.8],   # Stock 1: Mid cap, low beta
            [np.log(1e8), 1.5],   # Stock 2: Small cap, very high beta
        ][:num_stocks])
        
        # Categorical features: sector (0=Tech, 1=Finance, 2=Healthcare)
        static_categorical = np.array([[0], [1], [2]][:num_stocks])
        
        return {
            'target': stock_prices,
            'known_cov': known_cov,
            'unknown_cov': unknown_cov,
            'static_real': static_real,
            'static_categorical': static_categorical
        }
    
    def __len__(self):
        return self.data['target'].shape[0] - self.past_length - self.future_length
    
    def __getitem__(self, idx):
        past_start = idx
        past_end = idx + self.past_length
        future_start = past_end
        future_end = future_start + self.future_length
        
        return {
            "past_target": torch.tensor(
                self.data['target'][past_start:past_end], dtype=torch.float32
            ),
            "past_covariates_known_future": torch.tensor(
                self.data['known_cov'][past_start:past_end], dtype=torch.float32
            ),
            "past_covariates_unknown_future": torch.tensor(
                self.data['unknown_cov'][past_start:past_end], dtype=torch.float32
            ),
            "future_covariates_known": torch.tensor(
                self.data['known_cov'][future_start:future_end], dtype=torch.float32
            ),
            "output_target": torch.tensor(
                self.data['target'][future_start:future_end], dtype=torch.float32
            ),
            "static_features_real": torch.tensor(
                self.data['static_real'], dtype=torch.float32
            ),
            "static_features_categorical": torch.tensor(
                self.data['static_categorical'], dtype=torch.long
            ),
        }
    
    # Required properties
    @property
    def time_steps(self):
        return self.past_length
    
    @property
    def future_steps(self):
        return self.future_length
    
    @property
    def series_dim(self):
        return self.num_stocks
    
    @property
    def features_dim(self):
        return 1 + 3 + 2  # target + known_cov + unknown_cov
    
    @property
    def static_length(self):
        return 3  # 2 real + 1 categorical
    
    @property
    def static_categorical_cardinalities(self):
        return [3]  # 3 sectors
    
    @property
    def num_target_features(self):
        return 1
    
    @property
    def num_known_future_cov_features(self):
        return 3
    
    @property
    def num_unknown_future_cov_features(self):
        return 2
    
    @property
    def num_static_real_features(self):
        return 2
    
    @property
    def num_static_categorical_features(self):
        return 1

In [ ]:
# Create and test our custom dataset
stock_dataset = CustomStockDataset(num_stocks=3, total_days=300)

print(f"Custom dataset length: {len(stock_dataset)}")
print(f"Number of stocks: {stock_dataset.series_dim}")

# Examine a sample
sample = stock_dataset[0]
print("\nSample shapes:")
for key, tensor in sample.items():
    print(f"  {key}: {tensor.shape}")

## 5. Visualizing Custom Dataset

In [ ]:
# Visualize the custom stock dataset
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot stock prices
ax = axes[0, 0]
for stock_idx in range(3):
    past_prices = sample["past_target"][:, stock_idx, 0]
    future_prices = sample["output_target"][:, stock_idx, 0]
    
    time_past = range(len(past_prices))
    time_future = range(len(past_prices), len(past_prices) + len(future_prices))
    
    ax.plot(time_past, past_prices, label=f'Stock {stock_idx+1} (Past)', linewidth=2)
    ax.plot(time_future, future_prices, '--', label=f'Stock {stock_idx+1} (Future)', linewidth=2)

ax.axvline(x=len(past_prices)-0.5, color='gray', linestyle='-', alpha=0.7, linewidth=3)
ax.set_title('Stock Prices: Past vs Future')
ax.set_xlabel('Time Step')
ax.set_ylabel('Price')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot time features
ax = axes[0, 1]
time_features = sample["past_covariates_known_future"][:, 0, :]  # Stock 0
feature_names = ['Day of Week', 'Month', 'Month End']
for i in range(3):
    ax.plot(time_features[:, i], label=feature_names[i], alpha=0.8)
ax.set_title('Time Features (Known Covariates)')
ax.set_xlabel('Time Step')
ax.set_ylabel('Normalized Value')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot technical indicators
ax = axes[1, 0]
tech_features = sample["past_covariates_unknown_future"][:, 0, :]  # Stock 0
tech_names = ['SMA Ratio', 'Volatility']
for i in range(2):
    ax.plot(tech_features[:, i], label=tech_names[i], alpha=0.8)
ax.set_title('Technical Indicators (Unknown Covariates)')
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot static features
ax = axes[1, 1]
static_real = sample["static_features_real"]
static_cat = sample["static_features_categorical"]

# Bar plot for static features
stocks = [f'Stock {i+1}' for i in range(3)]
x = np.arange(len(stocks))

ax2 = ax.twinx()
bars1 = ax.bar(x - 0.15, static_real[:, 0], 0.3, label='Log Market Cap', alpha=0.7)
bars2 = ax.bar(x + 0.15, static_real[:, 1], 0.3, label='Beta', alpha=0.7)

sectors = ['Tech', 'Finance', 'Healthcare']
for i, (stock, sector) in enumerate(zip(stocks, sectors)):
    ax2.text(i, 0.5, sector, ha='center', va='center', fontweight='bold')

ax.set_xlabel('Stocks')
ax.set_ylabel('Static Real Features')
ax2.set_ylabel('Sector')
ax.set_title('Static Features')
ax.set_xticks(x)
ax.set_xticklabels(stocks)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Data Validation and Debugging

In [ ]:
def validate_dataset_sample(sample):
    """Helper function to validate a dataset sample."""
    required_keys = [
        "past_target", "past_covariates_known_future", 
        "past_covariates_unknown_future", "future_covariates_known",
        "output_target", "static_features_real", "static_features_categorical"
    ]
    
    print("Dataset Validation:")
    print("=" * 50)
    
    # Check required keys
    missing_keys = [key for key in required_keys if key not in sample]
    if missing_keys:
        print(f"❌ Missing keys: {missing_keys}")
    else:
        print("✅ All required keys present")
    
    # Check tensor types
    print("\nTensor Information:")
    for key, tensor in sample.items():
        print(f"  {key}:")
        print(f"    Shape: {tensor.shape}")
        print(f"    Dtype: {tensor.dtype}")
        print(f"    Range: [{tensor.min():.3f}, {tensor.max():.3f}]")
        
        # Check for NaN values
        if torch.isnan(tensor).any():
            print(f"    ⚠️  Contains NaN values")
        else:
            print(f"    ✅ No NaN values")
        print()
    
    # Check categorical dtypes
    if sample["static_features_categorical"].dtype != torch.long:
        print("❌ static_features_categorical should be torch.long")
    else:
        print("✅ Categorical features have correct dtype")

# Validate our custom dataset
validate_dataset_sample(stock_dataset[0])

## 7. Integration with FinTorch Models

Let's see how our dataset works with FinTorch models:

In [ ]:
# Example model integration (commented out as models may not be available in all environments)
"""
from fintorch.models.timeseries.tft.tft_module import TemporalFusionTransformerModule

# Create model with matching dimensions
model = TemporalFusionTransformerModule(
    number_of_past_inputs=stock_dataset.time_steps,
    horizon=stock_dataset.future_steps,
    num_past_target_features=stock_dataset.num_target_features,
    num_past_known_cov_features=stock_dataset.num_known_future_cov_features,
    num_past_unknown_cov_features=stock_dataset.num_unknown_future_cov_features,
    num_future_known_cov_features=stock_dataset.num_known_future_cov_features,
    num_static_real_features=stock_dataset.num_static_real_features,
    num_static_categorical_features=stock_dataset.num_static_categorical_features,
    static_categorical_cardinalities=stock_dataset.static_categorical_cardinalities,
    batch_size=32,
    device="cpu",
    series_selection_method="first"
)

# Test with a batch
dataloader = DataLoader(stock_dataset, batch_size=4, shuffle=False)
batch = next(iter(dataloader))

# Forward pass
with torch.no_grad():
    output = model(batch)
    print(f"Model output shape: {output.shape}")
"""

print("Model integration example (commented out)")
print("To use with FinTorch models:")
print("1. Initialize model with dataset feature dimensions")
print("2. Create DataLoader with appropriate batch size")
print("3. Pass batches directly to model - the standardized format is automatically compatible!")

## 8. Advanced Dataset Features

Let's explore some advanced features for real-world scenarios:

In [ ]:
# Example: Adding data normalization and missing value handling
class RobustStockDataset(CustomStockDataset):
    def __init__(self, *args, normalize=True, **kwargs):
        self.normalize = normalize
        super().__init__(*args, **kwargs)
        
        if self.normalize:
            self._compute_normalization_stats()
    
    def _compute_normalization_stats(self):
        """Compute normalization statistics."""
        # For unknown covariates
        unknown_cov = self.data['unknown_cov']
        self.unknown_mean = np.mean(unknown_cov, axis=(0, 1))
        self.unknown_std = np.std(unknown_cov, axis=(0, 1))
    
    def __getitem__(self, idx):
        sample = super().__getitem__(idx)
        
        if self.normalize:
            # Normalize unknown covariates
            unknown_cov = sample["past_covariates_unknown_future"]
            mean = torch.tensor(self.unknown_mean, dtype=torch.float32)
            std = torch.tensor(self.unknown_std, dtype=torch.float32)
            sample["past_covariates_unknown_future"] = (unknown_cov - mean) / (std + 1e-8)
        
        return sample

# Test robust dataset
robust_dataset = RobustStockDataset(num_stocks=2, total_days=200, normalize=True)
print(f"Robust dataset created with {len(robust_dataset)} samples")
print("Normalization applied to unknown covariates")

## Summary

In this tutorial, we've covered:

1. **Understanding the Format**: The standardized dictionary structure with clear feature separation
2. **Using Existing Datasets**: Working with SimpleSyntheticDataset and examining the data structure
3. **Creating Custom Datasets**: Building a stock price dataset from scratch
4. **Data Visualization**: Plotting different feature types to understand the data
5. **Validation**: Ensuring data quality and correct formatting
6. **Model Integration**: How the format works seamlessly with FinTorch models
7. **Advanced Features**: Adding normalization and robustness to datasets

### Key Benefits of the Standardized Format:

- **Clear Separation**: Different feature types are explicitly separated
- **Multi-Series Support**: Native support for multiple time series
- **Type Safety**: Proper tensor dtypes and shapes
- **Model Compatibility**: Works with all FinTorch time series models
- **Extensibility**: Easy to add new feature types and enhancements

### Next Steps:

1. Try creating your own dataset with real data
2. Experiment with different normalization strategies
3. Use the dataset with various FinTorch models (TFT, N-BEATS, etc.)
4. Implement custom preprocessing and feature engineering
5. Explore the other FinTorch tutorials for model-specific examples

### Resources:

- [FinTorch Documentation](https://fintorch.readthedocs.io/)
- [TFT Tutorial](../tft.ipynb)
- [Stock Tick Tutorial](../stocktick/stocktick.ipynb)
- [Market Data Tutorial](../marketdata/marketdata.ipynb)

Happy forecasting with FinTorch! 🚀